# 동형암호(FHE) 원리 실습 — 【답안지】 순수 파이썬

**서울여대 GSPP Privacy Scholar Camp · 이승환 (waLLLnut / LatticA)**

이 노트북은 **numpy조차 필요 없이** 파이썬 표준 라이브러리(`random`, `cmath`)만으로 동형암호의 핵심을 돌려봅니다.
코드는 `zip`·리스트 컴프리헨션 같은 **파이썬 전용 축약을 쓰지 않고 명시적 for-루프**로만 작성했습니다
(C/자바 등 다른 언어로도 그대로 옮길 수 있게 — 그래서 좀 깁니다).

> **두 가지 사용법**
> 1. 설명을 들으며 **Run All**.
> 2. `★ 실습` 셀의 `TRY_...`/`None` 슬롯을 바꿔 다시 실행.
>
> **주의:** 차원 $n=2$와 $[-10,10]$ 균등 샘플은 장난감 설정입니다.


## 0. 파라미터 설정
- 차원 $n=2$, 모듈러스 $q=10^6$, 스케일 $\Delta=10^2$, 시드 고정.


In [ ]:
import random, cmath   # 둘 다 파이썬 기본 내장 (설치 불필요)

q     = 10**6
n     = 2
Delta = 100
random.seed(42)

def center(x, mod=q):
    "값을 (-mod/2, mod/2] 범위로 (부호 있는 대표원)"
    return ((int(x) + mod // 2) % mod) - mod // 2

def dot(u, v):
    "두 벡터(리스트)의 내적 --- 명시적 루프"
    total = 0
    for i in range(len(u)):
        total = total + u[i] * v[i]
    return total

print("q =", q, ", n =", n, ", Delta =", Delta)

## 1. LWE 암호화 / phase / 복호화
확장키 $\bar{\mathbf s}=(1,s_1,s_2)$, 암호문 $\bar{\mathbf c}=(b,-a_1,-a_2)$,
**phase** $=\langle\bar{\mathbf c},\bar{\mathbf s}\rangle=\Delta m+e$. 반올림으로 $e$ 제거.


In [ ]:
def keygen():
    s = []
    for i in range(n):
        s.append(random.randint(-10, 10))
    return s

def sbar(s):
    r = [1]                      # 맨 앞 상수항 1
    for i in range(len(s)):
        r.append(s[i])
    return r                     # (1, s1, s2)

def encrypt(m, s, delta=Delta):
    a = []
    for i in range(n):
        a.append(random.randint(0, q - 1))
    e = random.randint(-10, 10)
    b = (dot(a, s) + e + delta * m) % q
    c = [b]                      # (b, -a1, -a2)
    for i in range(n):
        c.append((-a[i]) % q)
    return c

def phase(c, s):
    return center(dot(c, sbar(s)))

def decrypt(c, s, delta=Delta):
    return round(phase(c, s) / delta)

s = keygen()
print("비밀키 sk =", s, ", 확장키 sbar =", sbar(s))

In [ ]:
m1, m2 = 2, 3
c1 = encrypt(m1, s)
c2 = encrypt(m2, s)
print("암호문 c1 =", c1)
print("phase(c1) =", phase(c1, s), " (≈ Delta*m1 =", Delta * m1, ")  ->  복호:", decrypt(c1, s))
print("phase(c2) =", phase(c2, s), " (≈ Delta*m2 =", Delta * m2, ")  ->  복호:", decrypt(c2, s))
assert decrypt(c1, s) == m1 and decrypt(c2, s) == m2
print("OK: 암호화/복호화 동작")

## 2. 덧셈 — 암호문을 그대로 더한다
성분끼리 더하면 phase 도 그대로 더해져 $\Delta(m_1+m_2)+(e_1+e_2)$.


In [ ]:
c_add = []
for i in range(len(c1)):
    c_add.append((c1[i] + c2[i]) % q)

print("phase(c1+c2) =", phase(c_add, s))
print("복호:", decrypt(c_add, s), " (기대값 m1+m2 =", m1 + m2, ")")
assert decrypt(c_add, s) == m1 + m2
print("OK: 동형 덧셈")

### ★ 실습 1. 메시지와 오류를 바꾸면?
아래 세 값만 바꿔 실행. 오류 한계가 $\Delta/2=50$ 에 가까워지면 언제 복호가 흔들릴까요?


In [ ]:
TRY_M1, TRY_M2 = 4, -1
TRY_ERROR_BOUND = 10

trial_rng = random.Random(2026)

def encrypt_trial(m, s, error_bound):
    a = []
    for i in range(n):
        a.append(trial_rng.randint(0, q - 1))
    e = trial_rng.randint(-error_bound, error_bound)
    b = (dot(a, s) + e + Delta * m) % q
    c = [b]
    for i in range(n):
        c.append((-a[i]) % q)
    return c, e

tc1, te1 = encrypt_trial(TRY_M1, s, TRY_ERROR_BOUND)
tc2, te2 = encrypt_trial(TRY_M2, s, TRY_ERROR_BOUND)

tadd = []
for i in range(len(tc1)):
    tadd.append((tc1[i] + tc2[i]) % q)

got1, got2, got_add = decrypt(tc1, s), decrypt(tc2, s), decrypt(tadd, s)
print("개별 오류 (e1, e2):", (te1, te2), " / 합산 오류:", te1 + te2)
print("반올림 기준: |오류| <", Delta / 2)
print("개별 복호:", (got1, got2), " / 예상:", (TRY_M1, TRY_M2))
print("덧셈 복호:", got_add, " / 예상:", TRY_M1 + TRY_M2)
if (got1, got2, got_add) == (TRY_M1, TRY_M2, TRY_M1 + TRY_M2):
    print("결과: 성공")
else:
    print("결과: 복호 실패 — 오류와 Delta의 상대적 크기를 확인하세요")

## 3. 곱셈 — 비밀키와 암호문의 **텐서곱**
두 phase 의 곱 $=\langle\bar{\mathbf c}_1\otimes\bar{\mathbf c}_2,\ \bar{\mathbf s}\otimes\bar{\mathbf s}\rangle$.
새 암호문·키는 차원 9, phase 는 $\approx\Delta^2 m_1 m_2$ (스케일 $10^2\to10^4$).


In [ ]:
sb = sbar(s)

# 키 텐서곱 sbar ⊗ sbar  (이중 for 로 모든 (i,j) 곱)
t = []
for i in range(len(sb)):
    for j in range(len(sb)):
        t.append(sb[i] * sb[j])

# 암호문 텐서곱 c1 ⊗ c2  (mod q)
c_mul = []
for i in range(len(c1)):
    for j in range(len(c2)):
        c_mul.append((c1[i] * c2[j]) % q)

ph_mul = center(dot(c_mul, t))
print("텐서곱 암호문 차원:", len(c_mul), ", 텐서곱 키 차원:", len(t))
print("phase(곱) =", ph_mul, "  ≈ Delta^2 * m1*m2 =", Delta**2 * m1 * m2)
print("Delta^2 로 나눠 복호:", round(ph_mul / Delta**2), " (기대값", m1 * m2, ")")
assert round(ph_mul / Delta**2) == m1 * m2
print("OK: 동형 곱셈 (2차 암호문)")

### 선택 심화 3-1. 왜 하필 '텐서'인가
$\mu_1\mu_2=\bar{\mathbf s}^\top(\bar{\mathbf c}_1\bar{\mathbf c}_2^\top)\bar{\mathbf s}
=\langle\bar{\mathbf c}_1\otimes\bar{\mathbf c}_2,\ \bar{\mathbf s}\otimes\bar{\mathbf s}\rangle$.
외적 행렬의 이차형식과 텐서 내적이 같음을 확인.


In [ ]:
mu1, mu2 = phase(c1, s), phase(c2, s)

# 외적 행렬 C = c1 · c2^T  (이중 for)
C = []
for i in range(len(c1)):
    row = []
    for j in range(len(c2)):
        row.append(c1[i] * c2[j])
    C.append(row)

print("외적 행렬 C = c1·c2^T (3×3, 랭크 1):")
for i in range(len(C)):
    print("  ", C[i])

d = len(sb)

# (2) 이차형식  sbar^T C sbar
qf = 0
for i in range(d):
    for j in range(d):
        qf = qf + sb[i] * C[i][j] * sb[j]
qf = center(qf)

# (3) 텐서 내적  <vec(C), s⊗s>
Cvec = []
for i in range(d):
    for j in range(d):
        Cvec.append(C[i][j])
tens = center(dot(Cvec, t))

print()
print("(1) mu1 * mu2          =", mu1 * mu2)
print("(2) sbar^T C sbar      =", qf)
print("(3) <c1⊗c2, sbar⊗sbar> =", tens)
assert mu1 * mu2 == qf == tens
print("=> 셋 다 동일! 텐서곱은 '키에 대한 이차형식'을 편 것.")

### ★ 실습 2. 3개짜리(n=3)로 직접 해보기
`None` 슬롯 세 곳을 채우면 자동 검증. (확장키 차원 4, 텐서 차원 16)


In [ ]:
n3 = 3
rng3 = random.Random(7)

def keygen3():
    s = []
    for i in range(n3):
        s.append(rng3.randint(-10, 10))
    return s

def encrypt3(m, sk, delta=Delta):
    a = []
    for i in range(n3):
        a.append(rng3.randint(0, q - 1))
    e = rng3.randint(-10, 10)
    b = (dot(a, sk) + e + delta * m) % q
    c = [b]
    for i in range(n3):
        c.append((-a[i]) % q)
    return c

s3 = keygen3()
c1_3 = encrypt3(2, s3)   # 메시지 2
c2_3 = encrypt3(3, s3)   # 메시지 3

# ---- 채워야 할 슬롯 3개 (None 을 지우고 for-루프로 채우기) -----------------
sbar3 = [1]
for i in range(len(s3)):
    sbar3.append(s3[i])

t3 = []
for i in range(len(sbar3)):
    for j in range(len(sbar3)):
        t3.append(sbar3[i] * sbar3[j])

cmul3 = []
for i in range(len(c1_3)):
    for j in range(len(c2_3)):
        cmul3.append((c1_3[i] * c2_3[j]) % q)
# --------------------------------------------------------------------------

if sbar3 is None or t3 is None or cmul3 is None:
    print("슬롯(None)을 채운 뒤 다시 실행하세요.  힌트는 각 줄 주석 참고.")
    print("기대값 -> 확장키 차원 4, 텐서 차원 16, phase(곱) ≈", Delta**2 * 6)
else:
    add3 = []
    for i in range(len(c1_3)):
        add3.append((c1_3[i] + c2_3[i]) % q)
    ph_add = center(dot(add3, sbar3))
    ph_mul3 = center(dot(cmul3, t3))
    print("확장키 차원:", len(sbar3), "(기대 4) / 텐서 차원:", len(t3), "(기대 16)")
    print("덧셈 복호 :", round(ph_add / Delta),     "(기대 5)")
    print("곱셈 phase:", ph_mul3, "-> 복호:", round(ph_mul3 / Delta**2), "(기대 6)")
    assert len(sbar3) == n3 + 1 and len(t3) == (n3 + 1) ** 2
    assert round(ph_add / Delta) == 5 and round(ph_mul3 / Delta**2) == 6
    print("OK: n=3 에서도 암호화·덧셈·텐서곱 동작 🎉")

## 4. 키스위칭(리니어라이제이션) — 가젯 분해
큰 키 성분 $t_j$ 를 원래 키로 재암호화한 **KSK** 로, 2차 키를 다시 $\bar{\mathbf s}$ 로 줄입니다.


### ★ 실습 3. 가젯 분해를 직접 확인하기
`TRY_DIGIT_VALUE`만 바꾸고, 부호 있는 10진 자릿수를 먼저 예상해 보세요.


In [ ]:
Bg, L = 10, 6   # 10진 분해, 자릿수 6개 (10^6 = q)

def signed_digits(x):
    "x 를 부호 있는 10진 자릿수 L개로 분해"
    x = int(x) % q
    ds = []
    for _ in range(L):
        dgt = x % Bg
        if dgt > Bg // 2:
            dgt = dgt - Bg
        ds.append(dgt)
        x = (x - dgt) // Bg
    return ds

TRY_DIGIT_VALUE = 314159
try_digits = signed_digits(TRY_DIGIT_VALUE)

# 자릿수 재조합 (명시적 루프)
try_reconstructed = 0
for l in range(len(try_digits)):
    try_reconstructed = try_reconstructed + try_digits[l] * (Bg ** l)

print("원래 값 (mod q):", TRY_DIGIT_VALUE % q)
print("부호 있는 자릿수:", try_digits)
print("자릿수 재조합 (mod q):", try_reconstructed % q)
assert try_reconstructed % q == TRY_DIGIT_VALUE % q
print("OK: 가젯 분해 후 다시 같은 값")

#### KSK를 만들고 리니어라이제이션 실행

In [ ]:
def encrypt_raw(value, s):
    "스케일 없이 값 자체를 암호화 (KSK용)"
    a = []
    for i in range(n):
        a.append(random.randint(0, q - 1))
    e = random.randint(-10, 10)
    b = (dot(a, s) + e + int(value)) % q
    c = [b]
    for i in range(n):
        c.append((-a[i]) % q)
    return c

d_big = len(t)

# KSK: 큰 키 성분 t_j 를 각 자리 B^l 배로 암호화 (이중 루프)
KSK = {}
for j in range(1, d_big):
    row = []
    for l in range(L):
        row.append(encrypt_raw(t[j] * (Bg ** l), s))
    KSK[j] = row

def keyswitch(c_big):
    cprime = []
    for k in range(n + 1):
        cprime.append(0)
    cprime[0] = int(c_big[0]) % q          # 상수항(=1)은 b-슬롯으로
    for j in range(1, d_big):
        ds = signed_digits(c_big[j])
        for l in range(L):
            for k in range(n + 1):
                cprime[k] = (cprime[k] + ds[l] * KSK[j][l][k]) % q
    return cprime

c_ks = keyswitch(c_mul)
ph_ks = center(dot(c_ks, sb))
print("리니어라이제이션 후 차원:", len(c_ks), " (다시 원래 키", len(sb), "차원)")
print("phase =", ph_ks, " (곱 phase", ph_mul, "와 유사, 오류만 약간 증가)")
assert round(ph_ks / Delta**2) == m1 * m2
print("OK: 리니어라이제이션 (phase·메시지 보존)")

## 5. 장난감 리스케일 — $1/\Delta$ 로 나눠 원래 스케일로
암호문을 $\Delta$ 로 나눌 때 **모듈러스도 $q\to q/\Delta$ 로 함께** 줄입니다.


In [ ]:
q2 = q // Delta        # 새 모듈러스 10^4

c_rs = []
for i in range(len(c_ks)):
    c_rs.append(round(center(c_ks[i]) / Delta) % q2)

ph_rs = center(dot(c_rs, sb), q2)
print("리스케일 후 phase =", ph_rs, " (≈ Delta * m1*m2 =", Delta * m1 * m2, ")")
print("복호:", round(ph_rs / Delta), " (기대값", m1 * m2, ")")
assert round(ph_rs / Delta) == m1 * m2
print("OK: 리스케일 -> 원래 스케일의 정상 암호문 (곱셈 결과", m1 * m2, ")")

### 정리 (1부)
곱셈 → 리니어라이제이션 → 스케일·모듈러스 관리. 정확한 형식은 BGV·BFV·CKKS마다 다릅니다.


# 2부. NTT — 다항식 곱셈을 빠르게
단순화 링 $\mathbb Z_{17}[X]/(X^4-1)$ 에서 순환 합성곱을 스쿨북 = FFT = NTT 로 확인.


In [ ]:
P, N, w = 17, 4, 4

powers = []
for k in range(1, 5):
    powers.append(pow(w, k, P))
print("4^1..4^4 mod 17 =", powers, " (order =", N, "인 원시근)")

a = [1, 2, 3, 4]
b = [5, 6, 7, 8]
print("a =", a, ", b =", b)

## 6. 스쿨북 순환 합성곱 ($X^4\equiv1$ 이므로 인덱스 $\bmod 4$)

In [ ]:
def cyc_convol_schoolbook(a, b):
    res = []
    for i in range(N):
        res.append(0)
    for i in range(N):
        for j in range(N):
            k = (i + j) % N
            res[k] = (res[k] + a[i] * b[j]) % P
    return res

c_school = cyc_convol_schoolbook(a, b)
print("스쿨북 순환 합성곱 mod 17 =", c_school)

## 7. FFT 로 같은 결과 (복소 DFT를 `cmath` 로 직접, 명시적 루프)

In [ ]:
def dft(x):
    M = len(x)
    out = []
    for k in range(M):
        acc = 0
        for j in range(M):
            acc = acc + x[j] * cmath.exp(-2j * cmath.pi * k * j / M)
        out.append(acc)
    return out

def idft(X):
    M = len(X)
    out = []
    for j in range(M):
        acc = 0
        for k in range(M):
            acc = acc + X[k] * cmath.exp(2j * cmath.pi * k * j / M)
        out.append(acc / M)
    return out

Fa = dft(a)
Fb = dft(b)
prod = []
for k in range(N):
    prod.append(Fa[k] * Fb[k])

inv = idft(prod)
c_fft = []
for k in range(N):
    c_fft.append(round(inv[k].real) % P)

print("FFT 순환 합성곱 mod 17   =", c_fft)
assert c_fft == c_school
print("OK: 스쿨북 == FFT")

## 8. 원시근 4로 만든 **NTT 행렬** (정수 mod 17, 명시적 루프)

In [ ]:
winv = pow(w, -1, P)   # 4^{-1} mod 17
Ninv = pow(N, -1, P)

# NTT 행렬 W 와 역행렬 Wi (이중 루프)
Wm = []
for i in range(N):
    row = []
    for j in range(N):
        row.append(pow(w, (i * j) % N, P))
    Wm.append(row)

Wi = []
for i in range(N):
    row = []
    for j in range(N):
        row.append(pow(winv, (i * j) % N, P))
    Wi.append(row)

print("w^-1 =", winv, ", N^-1 =", Ninv)
print("NTT 행렬 W =")
for i in range(N):
    print("  ", Wm[i])

def ntt(x):
    out = []
    for i in range(N):
        acc = 0
        for j in range(N):
            acc = acc + Wm[i][j] * x[j]
        out.append(acc % P)
    return out

def intt(X):
    out = []
    for i in range(N):
        acc = 0
        for j in range(N):
            acc = acc + Wi[i][j] * X[j]
        out.append((Ninv * acc) % P)
    return out

Na = ntt(a)
Nb = ntt(b)
pw = []
for k in range(N):
    pw.append((Na[k] * Nb[k]) % P)
c_ntt = intt(pw)
print("NTT 순환 합성곱 mod 17   =", c_ntt)
assert c_ntt == c_school
print("OK: NTT == 스쿨북 == FFT")

## 9. 주파수 영역에서 mod 17 왕복

In [ ]:
A = ntt(a)

a_mod = []
for i in range(len(a)):
    a_mod.append(a[i] % P)

print("NTT(a)         =", A)
print("INTT(NTT(a))   =", intt(A), "  (원래 a =", a_mod, ")")
assert intt(A) == a_mod
print("OK: NTT <-> INTT 왕복 항등 (mod 17)")

### ★ 실습 4. 단위근을 바꾸면?
`1 <= TRY_W < 17`. `TRY_W=4`는 원시 4차 근. `TRY_W=2`면 차수·왕복이 어떻게 될지 예상 후 실행.


In [ ]:
TRY_W = 4

# 차수 = TRY_W 를 몇 번 곱해야 처음 1이 되는가 (명시적 탐색)
try_order = None
for k in range(1, P):
    if pow(TRY_W, k, P) == 1:
        try_order = k
        break

try_winv = pow(TRY_W, -1, P)

try_W = []
for i in range(N):
    row = []
    for j in range(N):
        row.append(pow(TRY_W, i * j, P))
    try_W.append(row)

try_Wi = []
for i in range(N):
    row = []
    for j in range(N):
        row.append(pow(try_winv, i * j, P))
    try_Wi.append(row)

# 정변환
try_A = []
for i in range(N):
    acc = 0
    for j in range(N):
        acc = acc + try_W[i][j] * a[j]
    try_A.append(acc % P)

# 역변환
try_back = []
for i in range(N):
    acc = 0
    for j in range(N):
        acc = acc + try_Wi[i][j] * try_A[j]
    try_back.append((Ninv * acc) % P)

a_mod = []
for i in range(len(a)):
    a_mod.append(a[i] % P)

print("TRY_W의 차수:", try_order, " / 필요한 차수:", N)
print("원시 N차 단위근인가?", try_order == N)
print("왕복 결과:", try_back, " / 원래 값:", a_mod)
print("왕복 성공?", try_back == a_mod)

### 정리 (2부)
- 순환 합성곱은 **스쿨북 = FFT = NTT** 로 모두 같은 결과.
- 이 노트북은 **numpy 없이, 그리고 `zip`·컴프리헨션 없이 명시적 루프만** 으로 전 과정을 돌렸습니다.

**수고하셨습니다! 🎉**


## 부록. 답안 (강사용)

**★ 실습 2 — n=3** (채운 슬롯, 명시적 for-루프)
```python
sbar3 = [1]
for i in range(len(s3)):
    sbar3.append(s3[i])

t3 = []
for i in range(len(sbar3)):
    for j in range(len(sbar3)):
        t3.append(sbar3[i] * sbar3[j])

cmul3 = []
for i in range(len(c1_3)):
    for j in range(len(c2_3)):
        cmul3.append((c1_3[i] * c2_3[j]) % q)
```
출력: 확장키 차원 4, 텐서 차원 16, 덧셈 복호 5, 곱셈 복호 6.

**★ 실습 1** 오류 한계를 키워 $|e_1+e_2|\ge\Delta/2=50$ 이면 덧셈 복호부터 실패.
**★ 실습 3** `TRY_DIGIT_VALUE` 를 무엇으로 두든 분해→재조합이 원래 값과 일치.
**★ 실습 4** `TRY_W=4` 성공(차수 4). `TRY_W=2` 는 차수 8(≠4)이라 원시 4차 근이 아니어서 왕복 실패.
